# Stage 3 configuration experiments (step 3-5) — ORL and CIFAR

This notebook runs the configuration experiments for ORL and CIFAR (MNIST is covered by Andrew)
and produces the tables and figures for report sections 3.5, 3.6 and 3.7.

* Code: `sai-changes` branch of `sampelts/RhinoTurtleCat` (shared configurable `Method_CNN`),
  plus four new files written by this notebook (none of the existing files are changed).
* Each configuration changes **one** setting of the dataset's baseline: depth, kernel size, padding,
  stride, pooling, hidden size, batch norm, dropout, loss function, optimizer.
* 10% of the supplied training split (stratified; 1 image per person for ORL) is held out as a
  **validation** split for comparing configurations. The supplied **test** split is evaluated once per run.
* Results are written to Google Drive after every run. **If Colab disconnects, re-run the same cells:
  finished runs are skipped.**

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Then run the cells from top to bottom.
The instructor's Stage 3 dataset is downloaded automatically from the course link.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: switch the runtime to T4 GPU')

## 1. Data and result folders
Results are saved to Google Drive after every run. The course dataset (`stage_3_data.zip`) is downloaded
from the instructor's public link. If that download fails, put the `ORL` and `CIFAR` files in a Drive folder
and set `MANUAL_DATA_DIR` to that folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/RhinoTurtleCat_stage3_ablation'  # results are saved here
MANUAL_DATA_DIR = ''   # leave empty to download automatically

import os
def find_data_dir(root):
    for dirpath, _, files in os.walk(root):
        if '__MACOSX' not in dirpath and 'ORL' in files and 'CIFAR' in files:
            return dirpath
    return None

if MANUAL_DATA_DIR:
    DATA_DIR = MANUAL_DATA_DIR
else:
    DATA_DIR = find_data_dir('/content/stage_3_data_raw')
    if DATA_DIR is None:
        !gdown -q 1-udd9jkldM1gEQ-OO-9HKXds42AiLOL6 -O /content/stage_3_data.zip
        !unzip -q -o /content/stage_3_data.zip -d /content/stage_3_data_raw
        DATA_DIR = find_data_dir('/content/stage_3_data_raw')

print('DATA_DIR =', DATA_DIR)
if not DATA_DIR or not all(os.path.exists(os.path.join(DATA_DIR, f)) for f in ('ORL', 'CIFAR')):
    raise RuntimeError('STOP: ORL/CIFAR data not found. Download stage_3_data.zip from the course page, unzip it, '
                     'put ORL and CIFAR in a Google Drive folder, set MANUAL_DATA_DIR to that folder, and re-run this cell.')
print('ORL found:', bool(DATA_DIR) and os.path.exists(os.path.join(DATA_DIR, 'ORL')),
      '| CIFAR found:', bool(DATA_DIR) and os.path.exists(os.path.join(DATA_DIR, 'CIFAR')))

## 2. Get the code

In [ ]:
REPO = '/content/RhinoTurtleCat'
if not os.path.exists(REPO):
    !git clone -q -b sai-changes https://github.com/sampelts/RhinoTurtleCat.git {REPO}
!cd {REPO} && git log -1 --format='%h %ad %s' --date=short
if not os.path.exists(f'{REPO}/local_code/stage_3_code/Method_CNN.py'):
    raise RuntimeError('STOP: the code was not downloaded (network problem?). Run !rm -rf /content/RhinoTurtleCat and re-run this cell.')

## 3. Write the four new files into the repository

In [ ]:
%%writefile /content/RhinoTurtleCat/local_code/stage_3_code/Method_CNN_MNIST_Ablation.py
'''
CNN for the MNIST hand-written digit dataset: 28x28 grayscale images, 10 classes.

Uses the shared, configurable Method_CNN (Sai) so MNIST can run the same
configuration experiments (step 3-5) as ORL and CIFAR. The baseline keeps the
architecture of the original MNIST model (Andrew): two conv blocks with 16 and
32 filters, a 128-unit dense layer, Adam lr 1e-3, batch size 64, 10 epochs.
Differences from the original MNIST run: inputs are standardized by
Dataset_Loader, and the shared classifier applies dropout 0.5.

Named *_Ablation so it does not collide with Method_CNN_MNIST.py on main.
'''

from local_code.stage_3_code.Method_CNN import Method_CNN


class Method_CNN_MNIST_Ablation(Method_CNN):
    input_shape = (1, 28, 28)
    num_classes = 10

    config = {
        **Method_CNN.config,
        'conv_channels': (16, 32),
        'hidden_dim': 128,
        'batch_size': 64,
        'max_epoch': 10,
    }


In [ ]:
%%writefile /content/RhinoTurtleCat/local_code/stage_3_code/Setting_Train_Val_Test.py
'''
Concrete SettingModule: carve a stratified validation set out of the supplied
training split, train on the rest, compare configurations on validation, and
evaluate the untouched supplied test split exactly once at the end.
'''

# Copyright (c) 2017-Current Jiawei Zhang <jiawei@ifmlab.org>
# License: TBD

from local_code.base_class.setting import setting
from sklearn.model_selection import train_test_split
import numpy as np


class Setting_Train_Val_Test(setting):
    val_fraction = 0.1
    split_seed = 0

    def split_train(self, X, y):
        n_classes = len(np.unique(y))
        # at least one validation image per class (ORL has only 9 training images per person)
        n_val = max(int(round(self.val_fraction * len(y))), n_classes)
        idx_train, idx_val = train_test_split(
            np.arange(len(y)), test_size=n_val, stratify=y, random_state=self.split_seed)
        return {'X': X[idx_train], 'y': y[idx_train]}, {'X': X[idx_val], 'y': y[idx_val]}

    def load_run_save_evaluate(self):
        # load dataset (already split into train/test by the instructor)
        data = self.dataset.load()
        train, val = self.split_train(data['train']['X'], data['train']['y'])
        test = data['test']

        # run MethodModule: Method_CNN draws its per-epoch curves on the 'test' entry,
        # so we hand it the validation split; the real test split stays untouched
        self.method.data = {'train': train, 'test': val}
        val_result = self.method.run()
        self.evaluate.data = val_result
        val_metrics = self.evaluate.evaluate()

        # final, single evaluation on the supplied test split
        test_result = {'pred_y': self.method.test(test['X']), 'true_y': test['y']}
        self.evaluate.data = test_result
        test_metrics = self.evaluate.evaluate()

        # save raw ResultModule (test predictions)
        self.result.data = test_result
        self.result.save()

        return {'val': val_metrics, 'test': test_metrics,
                'n_train': len(train['y']), 'n_val': len(val['y']), 'n_test': len(test['y'])}


In [ ]:
%%writefile /content/RhinoTurtleCat/script/stage_3_script/script_cnn_ablation.py
'''
Stage 3 configuration experiments (step 3-5) for MNIST, ORL and CIFAR in one runner.

Each configuration changes exactly one setting of the dataset's baseline CNN.
Models are trained on the supplied training split minus a stratified validation
split, compared on validation, and evaluated once on the untouched test split.

Runs are resumable: a run whose metrics.json already exists is skipped, so a
disconnected Colab session can simply re-run the same command.

usage:
    python script/stage_3_script/script_cnn_ablation.py --dataset MNIST
    python script/stage_3_script/script_cnn_ablation.py --dataset ORL --seeds 0 1 2
    python script/stage_3_script/script_cnn_ablation.py --dataset CIFAR --configs baseline deeper
'''

from pathlib import Path
import argparse
import csv
import json
import random
import sys
import time

PROJECT_ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from local_code.base_class.dataset import dataset
from local_code.stage_3_code.Dataset_Loader import Dataset_Loader
from local_code.stage_3_code.Method_CNN_MNIST_Ablation import Method_CNN_MNIST_Ablation
from local_code.stage_3_code.Method_CNN_ORL import Method_CNN_ORL
from local_code.stage_3_code.Method_CNN_CIFAR import Method_CNN_CIFAR
from local_code.stage_3_code.Result_Saver import Result_Saver
from local_code.stage_3_code.Evaluate_Accuracy import Evaluate_Accuracy
from local_code.stage_3_code.Setting_Train_Val_Test import Setting_Train_Val_Test
from local_code.stage_3_code.History_Saver import History_Saver
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch

# dataset name -> (method class, keep a single grayscale channel)
DATASETS = {
    'MNIST': (Method_CNN_MNIST_Ablation, False),
    'ORL': (Method_CNN_ORL, True),
    'CIFAR': (Method_CNN_CIFAR, False),
}

# configuration name -> what it changes (used in the report table)
CONFIG_DESCRIPTIONS = {
    'baseline': 'Baseline',
    'shallower': 'Depth: one fewer conv block',
    'deeper': 'Depth: one more conv block',
    'kernel_5': 'Kernel size 5x5 (padding 2)',
    'no_padding': 'Padding 0',
    'stride_2': 'Stride 2 conv instead of pooling',
    'avg_pool': 'Average pooling',
    'hidden_64': 'Hidden layer 64 units',
    'batch_norm': 'Add batch normalization',
    'no_dropout': 'No dropout',
    'margin_loss': 'Loss: multi-class margin',
    'sgd': 'Optimizer: SGD (lr 0.01, momentum 0.9)',
}


def make_configs(method_class):
    '''One-change-at-a-time overrides relative to the dataset's own baseline.'''
    base_channels = tuple(method_class.config['conv_channels'])
    return {
        'baseline': {},
        'shallower': {'conv_channels': base_channels[:-1]},
        'deeper': {'conv_channels': base_channels + (base_channels[-1] * 2,)},
        'kernel_5': {'kernel_size': 5, 'padding': 2},
        'no_padding': {'padding': 0},
        'stride_2': {'stride': 2, 'pool': None},
        'avg_pool': {'pool': 'avg'},
        'hidden_64': {'hidden_dim': 64},
        'batch_norm': {'batch_norm': True},
        'no_dropout': {'dropout': 0.0},
        'margin_loss': {'loss': 'multi_margin'},
        'sgd': {'optimizer': 'sgd', 'learning_rate': 0.01},
    }


class Cached_Dataset(dataset):
    '''Load the pickle once and hand the same arrays to every configuration.'''

    def __init__(self, loader):
        # the template's base class spells the attribute 'dataset_descrition'
        super().__init__(loader.dataset_name, getattr(loader, 'dataset_descrition', ''))
        self.loader = loader
        self.cache = None

    def load(self):
        if self.cache is None:
            self.cache = self.loader.load()
        return self.cache


class Val_History_Saver(History_Saver):
    '''Method_CNN labels its held-out curve "Test_*"; here that split is validation.'''

    def save(self):
        self.data = [{k.replace('Test_', 'Val_'): v for k, v in row.items()} for row in self.data]
        super().save()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def plot_curves(rows, path, title):
    epochs = [r['Epoch'] + 1 for r in rows]
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(10, 3.6))
    ax_loss.plot(epochs, [r['Train_Loss'] for r in rows], label='train')
    ax_loss.plot(epochs, [r['Val_Loss'] for r in rows], label='validation')
    ax_loss.set(xlabel='epoch', ylabel='loss', title='Loss')
    ax_acc.plot(epochs, [r['Train_Accuracy'] for r in rows], label='train')
    ax_acc.plot(epochs, [r['Val_Accuracy'] for r in rows], label='validation')
    ax_acc.set(xlabel='epoch', ylabel='accuracy', title='Accuracy')
    for ax in (ax_loss, ax_acc):
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def write_summary(out_dir):
    '''Rebuild summary.csv from every finished run, so re-runs never duplicate rows.'''
    rows = [json.loads(p.read_text())['summary'] for p in sorted(out_dir.glob('*/*/seed*/metrics.json'))]
    if not rows:
        return
    with open(out_dir / 'summary.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def run_one(dataset_name, cached_data, config_name, overrides, seed, args):
    method_class = DATASETS[dataset_name][0]
    run_dir = args.out_dir / dataset_name / config_name / f'seed{seed}'
    if (run_dir / 'metrics.json').exists():
        print(f'skip {dataset_name}/{config_name}/seed{seed} (already done)')
        return
    run_dir.mkdir(parents=True, exist_ok=True)
    set_seed(seed)

    overrides = dict(overrides)
    if args.epochs is not None:
        overrides['max_epoch'] = args.epochs

    # ---- object initialization section ------------------
    history_obj = Val_History_Saver('', '')
    history_obj.history_destination_folder_path = str(run_dir)
    history_obj.history_destination_file_name = 'history.csv'

    method_obj = method_class('CNN ' + dataset_name, config_name, **overrides)
    method_obj.history = history_obj

    result_obj = Result_Saver('saver', '')
    result_obj.result_destination_folder_path = str(run_dir) + '/'
    result_obj.result_destination_file_name = 'test_prediction_result'

    setting_obj = Setting_Train_Val_Test('train/val/test', '')
    setting_obj.val_fraction = args.val_fraction
    setting_obj.split_seed = 0  # same validation split for every configuration and seed

    evaluate_obj = Evaluate_Accuracy('accuracy, precision, recall, F1', '')

    # ---- running section ---------------------------------
    print(f'************ {dataset_name} / {config_name} / seed {seed} ************')
    setting_obj.prepare(cached_data, method_obj, result_obj, evaluate_obj)
    started = time.time()
    scores = setting_obj.load_run_save_evaluate()
    seconds = time.time() - started

    rows = history_obj.data
    plot_curves(rows, run_dir / 'learning_curve.png', f'{dataset_name} CNN ({config_name}, seed {seed})')
    config = {k: list(v) if isinstance(v, tuple) else v for k, v in method_obj.config.items()}
    summary = {
        'dataset': dataset_name,
        'config': config_name,
        'change': CONFIG_DESCRIPTIONS[config_name],
        'seed': seed,
        'epochs': config['max_epoch'],
        'parameters': sum(p.numel() for p in method_obj.parameters()),
        'n_train': scores['n_train'], 'n_val': scores['n_val'], 'n_test': scores['n_test'],
        'final_train_loss': rows[-1]['Train_Loss'],
        'final_val_loss': rows[-1]['Val_Loss'],
        'val_accuracy': scores['val']['accuracy'],
        'test_accuracy': scores['test']['accuracy'],
        'test_precision_macro': scores['test']['precision_macro'],
        'test_recall_macro': scores['test']['recall_macro'],
        'test_f1_macro': scores['test']['f1_macro'],
        'test_precision_weighted': scores['test']['precision_weighted'],
        'test_recall_weighted': scores['test']['recall_weighted'],
        'test_f1_weighted': scores['test']['f1_weighted'],
        'seconds': round(seconds, 1),
        'device': str(method_obj.device),
    }
    (run_dir / 'metrics.json').write_text(json.dumps(
        {'summary': summary, 'config': config,
         'val_metrics': {k: float(v) for k, v in scores['val'].items()},
         'test_metrics': {k: float(v) for k, v in scores['test'].items()}}, indent=2))
    print(f"val acc {summary['val_accuracy']:.4f} | test acc {summary['test_accuracy']:.4f} | {seconds:.0f}s")
    write_summary(args.out_dir)


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--dataset', choices=sorted(DATASETS), required=True)
    parser.add_argument('--configs', nargs='+', default=['all'],
                        help='configuration names, or "all" (default)')
    parser.add_argument('--seeds', nargs='+', type=int, default=[0])
    parser.add_argument('--epochs', type=int, default=None, help='override max_epoch for every configuration')
    parser.add_argument('--val-fraction', type=float, default=0.1)
    parser.add_argument('--data-dir', type=Path, default=PROJECT_ROOT / 'data' / 'stage_3_data')
    parser.add_argument('--out-dir', type=Path, default=PROJECT_ROOT / 'result' / 'stage_3_ablation')
    args = parser.parse_args()

    method_class, grayscale = DATASETS[args.dataset]
    configs = make_configs(method_class)
    names = list(configs) if args.configs == ['all'] else args.configs
    unknown = set(names) - set(configs)
    if unknown:
        parser.error('unknown configs: ' + ', '.join(sorted(unknown)) + '; choose from ' + ', '.join(configs))

    loader = Dataset_Loader(args.dataset, args.dataset + ' presplit dataset')
    loader.dataset_source_folder_path = str(args.data_dir) + '/'
    loader.dataset_source_file_name = args.dataset
    loader.grayscale = grayscale
    cached_data = Cached_Dataset(loader)

    for name in names:
        for seed in args.seeds:
            run_one(args.dataset, cached_data, name, configs[name], seed, args)
    print('************ Finish ************')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/RhinoTurtleCat/script/stage_3_script/summarize_ablation.py
'''
Turn result/stage_3_ablation/summary.csv into report-ready tables and figures.

Outputs (written next to summary.csv):
    ablation_table.md / .csv   one row per configuration, one column per dataset (section 3.7)
    baseline_metrics.md / .csv accuracy, precision, recall, F1 of each baseline (section 3.6)
    convergence.png            baseline train/validation loss per epoch, one panel per dataset (section 3.5)

usage:
    python script/stage_3_script/summarize_ablation.py
    python script/stage_3_script/summarize_ablation.py --out-dir /content/drive/MyDrive/stage3_ablation
'''

from pathlib import Path
import argparse
import csv
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parents[2]
DATASET_ORDER = ['MNIST', 'ORL', 'CIFAR']
CONFIG_ORDER = ['baseline', 'shallower', 'deeper', 'kernel_5', 'no_padding', 'stride_2', 'avg_pool',
                'hidden_64', 'batch_norm', 'no_dropout', 'margin_loss', 'sgd']


def fmt(mean, std, n):
    '''Percent with one decimal; add +/- std when several seeds were run.'''
    if pd.isna(mean):
        return '-'
    return f'{100 * mean:.1f} ± {100 * std:.1f}' if n > 1 else f'{100 * mean:.1f}'


def to_markdown(df):
    lines = ['| ' + ' | '.join(df.columns) + ' |', '|' + '---|' * len(df.columns)]
    lines += ['| ' + ' | '.join(str(v) for v in row) + ' |' for row in df.itertuples(index=False)]
    return '\n'.join(lines) + '\n'


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--out-dir', type=Path, default=PROJECT_ROOT / 'result' / 'stage_3_ablation')
    args = parser.parse_args()
    summary_path = args.out_dir / 'summary.csv'
    if not summary_path.exists():
        sys.exit(f'{summary_path} not found; run script_cnn_ablation.py first')

    runs = pd.read_csv(summary_path)
    datasets = [d for d in DATASET_ORDER if d in set(runs['dataset'])]
    stats = (runs.groupby(['dataset', 'config'])
             .agg(test_mean=('test_accuracy', 'mean'), test_std=('test_accuracy', 'std'),
                  val_mean=('val_accuracy', 'mean'), val_std=('val_accuracy', 'std'),
                  seeds=('seed', 'count'), change=('change', 'first'))
             .fillna({'test_std': 0.0, 'val_std': 0.0}))

    # ---- section 3.7: one row per configuration, one column per dataset -------------
    configs = [c for c in CONFIG_ORDER if c in set(runs['config'])]
    rows, csv_rows = [], []
    for config in configs:
        change = runs.loc[runs['config'] == config, 'change'].iloc[0]
        row, csv_row = {'Configuration': change}, {'config': config, 'change': change}
        for d in datasets:
            if (d, config) in stats.index:
                s = stats.loc[(d, config)]
                cell = fmt(s.test_mean, s.test_std, s.seeds)
                if config != 'baseline' and (d, 'baseline') in stats.index:
                    delta = 100 * (s.test_mean - stats.loc[(d, 'baseline')].test_mean)
                    cell += f' ({delta:+.1f})'
                row[f'{d} test acc %'] = cell
                row[f'{d} val acc %'] = fmt(s.val_mean, s.val_std, s.seeds)
                csv_row.update({f'{d}_test_acc': s.test_mean, f'{d}_test_std': s.test_std,
                                f'{d}_val_acc': s.val_mean, f'{d}_val_std': s.val_std, f'{d}_seeds': s.seeds})
            else:
                row[f'{d} test acc %'] = row[f'{d} val acc %'] = '-'
        rows.append(row)
        csv_rows.append(csv_row)
    table = pd.DataFrame(rows)
    # compact version for the report: test accuracy only
    compact = table[['Configuration'] + [f'{d} test acc %' for d in datasets]]
    note = ('\nTest accuracy in %, difference from the baseline in parentheses; '
            '"± x" is the standard deviation over seeds. Configurations were compared on the '
            'validation split (full table below); the supplied test split was evaluated once per run.\n')
    (args.out_dir / 'ablation_table.md').write_text(
        '## Configuration experiments (compact, for the report)\n\n' + to_markdown(compact) + note +
        '\n## Full table (test and validation accuracy)\n\n' + to_markdown(table))
    pd.DataFrame(csv_rows).to_csv(args.out_dir / 'ablation_table.csv', index=False)

    # ---- section 3.6: baseline accuracy / precision / recall / F1 -----------------------
    base = runs[runs['config'] == 'baseline']
    metric_cols = ['test_accuracy', 'test_precision_macro', 'test_recall_macro', 'test_f1_macro']
    perf_rows = []
    for d in datasets:
        part = base[base['dataset'] == d]
        if part.empty:
            continue
        n = len(part)
        perf_rows.append({'Dataset': d, 'Seeds': n,
                          **{c.replace('test_', '').replace('_macro', ' (macro)').capitalize() + ' %':
                             fmt(part[c].mean(), part[c].std() if n > 1 else 0.0, n) for c in metric_cols}})
    perf = pd.DataFrame(perf_rows)
    (args.out_dir / 'baseline_metrics.md').write_text(
        '## Baseline model performance on the supplied test split\n\n' + to_markdown(perf))
    perf.to_csv(args.out_dir / 'baseline_metrics.csv', index=False)

    # ---- section 3.5: convergence plot, x = epoch, y = loss -----------------------------
    fig, axes = plt.subplots(1, len(datasets), figsize=(4 * len(datasets), 3.2), squeeze=False)
    for ax, d in zip(axes[0], datasets):
        seed_dirs = sorted((args.out_dir / d / 'baseline').glob('seed*/history.csv'))
        if not seed_dirs:
            ax.set_visible(False)
            continue
        with open(seed_dirs[0]) as f:
            hist = list(csv.DictReader(f))
        epochs = [int(r['Epoch']) + 1 for r in hist]
        ax.plot(epochs, [float(r['Train_Loss']) for r in hist], label='train loss')
        ax.plot(epochs, [float(r['Val_Loss']) for r in hist], '--', label='validation loss')
        ax.set(title=f'{d} CNN (baseline)', xlabel='Training epoch', ylabel='Cross-entropy loss')
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig(args.out_dir / 'convergence.png', dpi=200)
    plt.close(fig)

    print((args.out_dir / 'ablation_table.md').read_text())
    print((args.out_dir / 'baseline_metrics.md').read_text())
    print('figures and tables written to', args.out_dir)


if __name__ == '__main__':
    main()


## 4. Smoke test (about a minute)
Runs the ORL baseline for one epoch into a scratch folder, to catch path or data problems before the long runs.

In [ ]:
!cd {REPO} && python -u script/stage_3_script/script_cnn_ablation.py --dataset ORL --configs baseline --epochs 1 \
    --data-dir "{DATA_DIR}" --out-dir /content/smoke_test 2>&1 | grep --line-buffered -v -E "^Epoch: |method running|start training|start testing|saving|history saved|conv output size"

## 5. Run the experiments
12 configurations per dataset. Defaults: ORL 50 epochs × 3 seeds (its test split has only
40 images, so seeds show how much results move), CIFAR 20 epochs. The console labels the held-out curve
`Test_*` (name hard-coded in `Method_CNN`); in this setup it is the **validation** split, and saved files say `Val_*`.

In [ ]:
!cd {REPO} && python -u script/stage_3_script/script_cnn_ablation.py --dataset ORL --seeds 0 1 2 \
    --data-dir "{DATA_DIR}" --out-dir "{OUT_DIR}" 2>&1 | grep --line-buffered -v -E "^Epoch: |method running|start training|start testing|saving|history saved|conv output size"

In [ ]:
import glob
done = glob.glob(f'{OUT_DIR}/ORL/*/seed*/metrics.json')
print(f'ORL: {len(done)} finished runs saved')
if not done:
    raise RuntimeError('STOP: no ORL run was saved. Scroll up to the previous cell for the error message.')

In [ ]:
!cd {REPO} && python -u script/stage_3_script/script_cnn_ablation.py --dataset CIFAR \
    --data-dir "{DATA_DIR}" --out-dir "{OUT_DIR}" 2>&1 | grep --line-buffered -v -E "^Epoch: |method running|start training|start testing|saving|history saved|conv output size"

In [ ]:
import glob
done = glob.glob(f'{OUT_DIR}/CIFAR/*/seed*/metrics.json')
print(f'CIFAR: {len(done)} finished runs saved')
if not done:
    raise RuntimeError('STOP: no CIFAR run was saved. Scroll up to the previous cell for the error message.')

## 6. Tables and figures for the report

In [ ]:
!cd {REPO} && python script/stage_3_script/summarize_ablation.py --out-dir "{OUT_DIR}" > /dev/null
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{OUT_DIR}/baseline_metrics.md').read()))
display(Markdown(open(f'{OUT_DIR}/ablation_table.md').read()))
display(Image(f'{OUT_DIR}/convergence.png'))

## 7. Download everything (optional; it is already on Drive)
Commit `summary.csv`, the tables, `convergence.png` and the per-run `metrics.json` / `history.csv` to the repo
so the report numbers can be traced. Note: the `sai-changes` `.gitignore` ignores `result/`, so use `git add -f`.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/stage3_ablation_results', 'zip', OUT_DIR)
files.download('/content/stage3_ablation_results.zip')